# MP10 — SSIM Evaluation Tool

Expanded executable lesson. Prerequisites: earlier image-processing and array lessons. Learning goals: derive the calculation, predict edge cases, implement it, and interpret controlled real-image evidence. Learner mastery is not asserted.

## Setup and conventions
From repository root run `python -m pip install -e ".[dev]"`. Run `python scripts/metric_experiments.py` after downloading the pinned training scene with the portable quickstart. The notebook reads the committed numerical record; it does not silently download imagery. Arrays are HWC float64 for metrics.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from rssr.metric_lab import mse_psnr,ssim_map


## Mathematics and worked example
SSIM compares local luminance and local second-order statistics. For a window of n pixels, mu_x=sum(x)/n, variance_x=sum((x-mu_x)^2)/n, and covariance_xy=sum((x-mu_x)(y-mu_y))/n. We use population moments, uniform 7×7 windows, no padding, one score per band/window, and average these equally. The valid map has shape (H-6,W-6,C). This differs from implementations using Gaussian weights or sample moments.

SSIM=((2 mu_x mu_y+C1)(2 covariance+C2))/((mu_x²+mu_y²+C1)(variance_x+variance_y+C2)), with C1=(.01 L)² and C2=(.03 L)². Constants stabilize nearly flat regions. For constant windows x=.5,y=.6,L=1, variance and covariance vanish, the second ratio is one, and SSIM=(.6+.0001)/(.25+.36+.0001), approximately .98361. Identical windows have score one. Negative covariance may yield negative SSIM; it is not a probability.

Two errors can have equal MSE and PSNR but different spatial structure. The recorded experiment scales a noise realization to exactly the blur MSE; SSIM distinguishes their local statistics. This does not make SSIM a universal perceptual or scientific quality oracle. Multi-band averaging also hides which band failed.

In [ ]:
x=np.full((9,10,2),.5); y=np.full_like(x,.6)
s=ssim_map(x,y)
expected=(.6+.0001)/(.61+.0001)
assert s.shape==(3,4,2)
np.testing.assert_allclose(s,expected)
np.testing.assert_allclose(ssim_map(x,x),1)
print(s.mean())

## Recorded real-data experiment
The source checksum, acquisition date, license, normalization, seed and settings are in the record below. The input is a 64×64 training crop of real four-band NAIP aerial imagery; these are synthetic distortions, not satellite sensor measurements. Reproduce with `python scripts/metric_experiments.py`.

In [ ]:
record=json.loads((root/'reports/mp09-mp11-experiments.json').read_text())
print(record['source']['id'],record['source']['sha256'])
print(record['protocol'])

## Practice
Predict before running: (1) Map dimensions for 16×20×4 and window 7? (2) Why exclude a window containing nodata? (3) Is default-library SSIM comparable without checking its settings?

## Worked solutions
(1) 10×14×4. (2) Its moments would mix observations with missing-data fill values; mask whole windows, not only their centers. (3) No: range, window, weights, covariance normalization, border and band aggregation must agree.

In [ ]:
assert ssim_map(np.ones((16,20,4)),np.ones((16,20,4))).shape==(10,14,4)
rows={r['case']:r for r in record['rows']}
assert np.isclose(rows['blur']['mse'],rows['matched-MSE noise']['mse'])
print('Equal MSE, different SSIM:',rows['blur']['ssim'],rows['matched-MSE noise']['ssim'])

## Figure and interpretation
See [dashboard](../reports/mp09-mp11-dashboard.png). Blur and matched-MSE noise have equal PSNR but different SSIM. All error maps share a scale. Contrast and blur affect this low-texture crop less than the .01 noise case; this observation is not a universal ranking. Reflect: explain a failure mode that both metrics could miss.

## References and limitations
[Official scikit-image metric documentation](https://scikit-image.org/docs/stable/api/skimage.metrics.html), checked 2026-10-19, documents explicit ranges and SSIM settings. Our NumPy implementation uses declared uniform population moments and valid windows; it does not claim default-package equivalence. No alignment correction, learned perceptual metric or independent-scene claim is made.